<a href="https://colab.research.google.com/github/Kommmi/Qaos/blob/main/QuantumCircuits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install cirq
import cirq
import numpy as np
from IPython.display import clear_output
clear_output()

print("Module ready to go :)")

Module ready to go :)


1. Creating a qubit

This does not specify its wavefunction. It creates a label for the qubit. Argument indicates the position of the qubit. "qubit" - object that identifies which qubit operations should act on.



In [ ]:
q0 = cirq.LineQubit(0)
q1 = cirq.LineQubit(1)

# Create two qubits at once, in a line.
q0, q1 = cirq.LineQubit.range(2)

2. Gates: specify the operations.

Example: Preparing the wavefunction of the qubit

Start from state $|0\rangle$ apply $R_y(\theta)$ followed by $R_z(\phi)$. This give the general state of the qubit,

$$|\psi⟩ = \cos \frac{\theta}{2}|0⟩ + e^{i\phi}\sin \frac{\theta}{2}|1⟩$$

In [ ]:
theta0 = np.pi / 2
phi0 = np.pi / 3

theta1 = np.pi / 4
phi1 = np.pi / 5

circuit = cirq.Circuit(
    cirq.ry(theta0)(q0),
    cirq.rz(phi0)(q0),
    cirq.ry(theta1)(q1),
    cirq.rz(phi1)(q1)
)

print(circuit)

0: ───Ry(0.5π)────Rz(0.333π)───

1: ───Ry(0.25π)───Rz(0.2π)─────


3. Circuit: Ordered collection of quantum operations

A circuit doesn't itself perform the calculations. It is a description of quantum evolution. It has not evolved the wavefunction. The gates specify the operations.

In [ ]:
circuit = cirq.Circuit(
    cirq.ry(theta0)(q0),
    cirq.rz(phi0)(q0),
    cirq.ry(theta1)(q1),
    cirq.rz(phi1)(q1),
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, q1, key="result")
)

print(circuit)

0: ───Ry(0.5π)────Rz(0.333π)───H───@───M('result')───
                                   │   │
1: ───Ry(0.25π)───Rz(0.2π)─────────X───M─────────────


Appending to the circuit

In [ ]:
circuit = cirq.Circuit()

circuit.append(cirq.H(q0))
circuit.append(cirq.CNOT(q0, q1))

print(circuit)

0: ───H───@───
          │
1: ───────X───


 4. Moment: contains operations that occur during the same abstract time step

In [ ]:
circuit = cirq.Circuit(
    cirq.Moment([
        cirq.H(q0),
        cirq.X(q1)
    ]),
    cirq.Moment([
        cirq.CNOT(q0, q1)
    ])
)

print(circuit)

print("\nMoment-0\n")
print(circuit[0])

print("\nMoment-1\n")
print(circuit[1])

print("\nLength of Circuit or Circuit Depth: ",len(circuit))

0: ───H───@───
          │
1: ───X───X───

Moment-0

  ╷ 0 1
╶─┼─────
0 │ H X
  │

Moment-1

  ╷ 0 1
╶─┼─────
0 │ @─X
  │

Length of Circuit or Circuit Depth:  2


5. Simulator: executes the instructions specified in the circuit

Single Qubit

In [ ]:
q = cirq.LineQubit(0)

circuit = cirq.Circuit(
    cirq.H(q)
)

simulator = cirq.Simulator()
result = simulator.simulate(circuit)
print(circuit)

print(result.final_state_vector)

0: ───H───
[0.70710677+0.j 0.70710677+0.j]


Multiple Qubit

In [ ]:
q0, q1 = cirq.LineQubit.range(2)

circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1)
)

result = simulator.simulate(circuit)

print(result.final_state_vector)

[0.70710677+0.j 0.        +0.j 0.        +0.j 0.70710677+0.j]


In [ ]:
for step in simulator.simulate_moment_steps(circuit):
    print(step.state_vector())

[0.70710677+0.j 0.        +0.j 0.70710677+0.j 0.        +0.j]
[0.70710677+0.j 0.        +0.j 0.        +0.j 0.70710677+0.j]


6. Simulate vs Run



<details>
<summary><b>CIRCUIT Flow Diagram</b></summary>

```
             CIRCUIT
                │
                │ instructions
                ↓
          ┌───────────┐
          │ Simulator │
          └───────────┘
             ↙     ↘
            ↙       ↘
     simulate()     run()
         │            │
         ↓            ↓
   wavefunction    measurement
     / state          samples
                       │
                       ↓
                  many shots
```

</details>

In [ ]:
circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, q1, key="m")
)

print(circuit)

result = simulator.run(
    circuit,
    repetitions=1000
)

0: ───H───@───M('m')───
          │   │
1: ───────X───M────────


In [ ]:
import cirq

q0, q1 = cirq.LineQubit.range(2)

circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, q1, key="m")
)

simulator = cirq.Simulator()

result = simulator.run(
    circuit,
    repetitions=20
)

print("Circuit:")
print(circuit)

print("\nRaw measurements:")
print(result.measurements["m"].astype(int))

print("\nHistogram:")
print(result.histogram(key="m"))

Circuit:
0: ───H───@───M('m')───
          │   │
1: ───────X───M────────

Raw measurements:
[[1 1]
 [1 1]
 [1 1]
 [0 0]
 [0 0]
 [0 0]
 [1 1]
 [1 1]
 [1 1]
 [0 0]
 [1 1]
 [1 1]
 [0 0]
 [0 0]
 [1 1]
 [0 0]
 [1 1]
 [0 0]
 [0 0]
 [0 0]]

Histogram:
Counter({3: 10, 0: 10})


# Quantum Kicked Top Simulation

In [ ]:
theta = np.pi / 2 + 0.5
phi = np.pi / 2

circuit = cirq.Circuit(
    cirq.ry(theta0)(q0),
    cirq.rz(phi0)(q0),
    cirq.ry(theta1)(q1),
    cirq.rz(phi1)(q1)
)

print(circuit)

In [ ]:

# --------------------------------
# Parameters
# --------------------------------

nqubits = 3
kappa = 0.5

# --------------------------------
# Qubits
# --------------------------------

q0, q1, q2 = cirq.LineQubit.range(nqubits)

# --------------------------------
# Initial State Preparation
# --------------------------------

theta = np.pi / 2 + 0.5
phi = np.pi / 2
circuit = cirq.Circuit(
    cirq.ry(theta)(q0),
    cirq.rz(phi)(q0),
    cirq.ry(theta)(q1),
    cirq.rz(phi)(q1),
    cirq.ry(theta)(q2),
    cirq.rz(phi)(q2),
)


# --------------------------------
# One kicked-top Floquet step
# --------------------------------

kick_circuit = cirq.Circuit()


# Collective y rotation:
# exp[-i (pi/2) Jy]
kick_circuit.append([
    cirq.ry(np.pi / 2)(q0),
    cirq.ry(np.pi / 2)(q1),
    cirq.ry(np.pi / 2)(q2),
])

# Twisting:
# exp[-i kappa/(2j) Jz^2]
#
# For 3 qubits:
# exp[-i (kappa/6) Zi Zj]
#
# Cirq's ZZPowGate implements this up to a global phase.

zz_exponent = kappa / (3 * np.pi)

kick_circuit.append(cirq.ZZPowGate(exponent=zz_exponent)(q0, q1))
kick_circuit.append(cirq.ZZPowGate(exponent=zz_exponent)(q0, q2))
kick_circuit.append(cirq.ZZPowGate(exponent=zz_exponent)(q1, q2))

print(kick_circuit)

0: ───Ry(0.5π)───ZZ─────────ZZ────────────────────
                 │          │
1: ───Ry(0.5π)───ZZ^0.053───┼──────────ZZ─────────
                            │          │
2: ───Ry(0.5π)──────────────ZZ^0.053───ZZ^0.053───


## Simulation

In [ ]:
psi = np.zeros(8, dtype=complex)
psi[0] = 1.0

psi_steps = [psi.copy()]

for n in range(100):

    result = simulator.simulate(
        circuit,
        initial_state=psi
    )

    psi = result.final_state_vector.copy()

    psi_steps.append(psi)

## Run

In [ ]:
import cirq
import numpy as np

q0, q1, q2 = cirq.LineQubit.range(3)

system = q0
environment = [q1, q2]

circuit = cirq.Circuit()

# Apply one QKT kick
circuit += kick_circuit

# Measure environment
circuit.append(
    cirq.measure(q1, q2, key="env")
)

print(circuit)

0: ───Ry(0.5π)───ZZ─────────ZZ───────────────────────────────
                 │          │
1: ───Ry(0.5π)───ZZ^0.053───┼──────────ZZ─────────M('env')───
                            │          │          │
2: ───Ry(0.5π)──────────────ZZ^0.053───ZZ^0.053───M──────────


In [3]:
import cirq
import numpy as np
from collections import Counter


# ============================================================
# 1. QUBITS
# ============================================================

q0, q1, q2 = cirq.LineQubit.range(3)

system = q0
environment = [q1, q2]


# ============================================================
# 2. INITIAL STATE PREPARATION
# ============================================================

def initial_state_circuit(qubits, theta, phi):
    """
    Prepare the same single-qubit state on every qubit:

        |psi(theta,phi)> =
            cos(theta/2)|0>
            + exp(i phi) sin(theta/2)|1>

    starting from |000>.
    """

    circuit = cirq.Circuit()

    for q in qubits:
        circuit.append(cirq.ry(theta)(q))
        circuit.append(cirq.rz(phi)(q))

    return circuit


# ============================================================
# 3. ONE QUANTUM-KICKED-TOP FLOQUET STEP
# ============================================================

def qkt_kick_circuit(qubits, kappa):
    """
    One Floquet step

        U_F = exp[-i kappa/(2j) Jz^2]
              exp[-i pi/2 Jy]

    for L qubits, j = L/2.

    Global phases are irrelevant.
    """

    L = len(qubits)

    circuit = cirq.Circuit()

    # --------------------------------------------------------
    # Collective rotation exp[-i pi/2 Jy]
    # --------------------------------------------------------

    circuit.append(
        cirq.ry(np.pi / 2)(q)
        for q in qubits
    )

    # --------------------------------------------------------
    # Twisting exp[-i kappa/(2j) Jz^2]
    #
    # Jz^2 = 1/4 [L I + 2 sum_{i<j} Zi Zj]
    #
    # Since j = L/2:
    #
    # pair term = exp[-i kappa/(2L) Zi Zj]
    #
    # ZZPowGate(t) ~ exp[-i pi*t/2 ZZ]
    #
    # Therefore t = kappa/(L*pi)
    # --------------------------------------------------------

    zz_exponent = kappa / (L * np.pi)

    for i in range(L):
        for j in range(i + 1, L):

            circuit.append(
                cirq.ZZPowGate(
                    exponent=zz_exponent
                )(qubits[i], qubits[j])
            )

    return circuit


# ============================================================
# 4. BUILD CIRCUIT FOR n KICKS
# ============================================================

def evolution_circuit(
    qubits,
    theta,
    phi,
    kappa,
    nkicks
):
    """
    Prepare the initial state and apply nkicks QKT kicks.
    """

    circuit = initial_state_circuit(
        qubits,
        theta,
        phi
    )

    kick = qkt_kick_circuit(
        qubits,
        kappa
    )

    for _ in range(nkicks):
        circuit += kick

    return circuit


# ============================================================
# 5. ADD ENVIRONMENT + SYSTEM MEASUREMENTS
# ============================================================

def tomography_circuit(
    base_circuit,
    system,
    environment,
    basis
):
    """
    Construct a measurement circuit for conditional tomography.

    Environment is measured in the computational basis.

    System is measured in X, Y, or Z basis.
    """

    circuit = base_circuit.copy()

    # --------------------------------------------------------
    # Measure environment
    # --------------------------------------------------------

    circuit.append(
        cirq.measure(
            *environment,
            key="env"
        )
    )

    # --------------------------------------------------------
    # Rotate system into desired tomography basis
    # --------------------------------------------------------

    if basis == "X":

        circuit.append(
            cirq.H(system)
        )

    elif basis == "Y":

        circuit.append(
            cirq.S(system) ** -1
        )

        circuit.append(
            cirq.H(system)
        )

    elif basis == "Z":

        pass

    else:

        raise ValueError(
            "basis must be X, Y, or Z"
        )

    # --------------------------------------------------------
    # Measure system
    # --------------------------------------------------------

    circuit.append(
        cirq.measure(
            system,
            key="sys"
        )
    )

    return circuit


# ============================================================
# 6. CONVERT ENVIRONMENT BITS -> INTEGER LABEL
# ============================================================

def bits_to_integer(bits):
    """
    Example:

        [0,0] -> 0
        [0,1] -> 1
        [1,0] -> 2
        [1,1] -> 3
    """

    value = 0

    for bit in bits:
        value = (value << 1) | int(bit)

    return value


# ============================================================
# 7. RUN ONE TOMOGRAPHY BASIS
# ============================================================

def run_basis(
    simulator,
    circuit,
    repetitions
):
    """
    Run circuit and return environment outcomes and
    system measurement outcomes.
    """

    result = simulator.run(
        circuit,
        repetitions=repetitions
    )

    env = result.measurements["env"].astype(int)

    sys = (
        result.measurements["sys"]
        .astype(int)
        .flatten()
    )

    env_labels = np.array([
        bits_to_integer(row)
        for row in env
    ])

    return env_labels, sys


# ============================================================
# 8. COMPUTE <X>, <Y>, OR <Z> CONDITIONED ON ENVIRONMENT
# ============================================================

def conditional_expectation(
    env_labels,
    sys_results,
    env_state
):
    """
    Calculate

        <sigma>_e = P(0|e) - P(1|e)

    for one environment outcome e.
    """

    mask = env_labels == env_state

    measurements = sys_results[mask]

    if len(measurements) == 0:
        return np.nan

    p0 = np.mean(measurements == 0)
    p1 = np.mean(measurements == 1)

    return p0 - p1


# ============================================================
# 9. RECONSTRUCT Q^S AT ONE KICK NUMBER
# ============================================================

def measure_GQS_at_kick(
    simulator,
    qubits,
    system,
    environment,
    theta0,
    phi0,
    kappa,
    nkicks,
    repetitions=10000
):
    """
    Reconstruct

        Q^S(n) = {lambda_e, theta_e, phi_e}

    after nkicks.

    Returns one entry for every observed environment state.
    """

    # --------------------------------------------------------
    # Evolution to time n
    # --------------------------------------------------------

    base = evolution_circuit(
        qubits,
        theta0,
        phi0,
        kappa,
        nkicks
    )

    # --------------------------------------------------------
    # Create three independent tomography experiments
    # --------------------------------------------------------

    cx = tomography_circuit(
        base,
        system,
        environment,
        "X"
    )

    cy = tomography_circuit(
        base,
        system,
        environment,
        "Y"
    )

    cz = tomography_circuit(
        base,
        system,
        environment,
        "Z"
    )

    # --------------------------------------------------------
    # Run experiments
    # --------------------------------------------------------

    env_x, sys_x = run_basis(
        simulator,
        cx,
        repetitions
    )

    env_y, sys_y = run_basis(
        simulator,
        cy,
        repetitions
    )

    env_z, sys_z = run_basis(
        simulator,
        cz,
        repetitions
    )

    # --------------------------------------------------------
    # Environment probabilities lambda_e
    #
    # Use all three tomography experiments to improve
    # statistics for lambda.
    # --------------------------------------------------------

    all_env = np.concatenate([
        env_x,
        env_y,
        env_z
    ])

    counts = Counter(all_env)

    total = len(all_env)

    # Number of possible environment outcomes
    dE = 2 ** len(environment)

    Q_S = []

    # --------------------------------------------------------
    # Conditional tomography
    # --------------------------------------------------------

    for e in range(dE):

        lam = counts[e] / total

        if lam == 0:
            continue

        x = conditional_expectation(
            env_x,
            sys_x,
            e
        )

        y = conditional_expectation(
            env_y,
            sys_y,
            e
        )

        z = conditional_expectation(
            env_z,
            sys_z,
            e
        )

        bloch = np.array(
            [x, y, z],
            dtype=float
        )

        # ----------------------------------------------------
        # Finite-shot tomography may give |r| != 1.
        #
        # For an ideal projected pure state, normalize
        # the Bloch vector before converting to theta, phi.
        # ----------------------------------------------------

        r = np.linalg.norm(bloch)

        if r > 0:
            bloch_pure = bloch / r
        else:
            bloch_pure = bloch

        x_p, y_p, z_p = bloch_pure

        z_p = np.clip(
            z_p,
            -1.0,
            1.0
        )

        theta = np.arccos(z_p)

        phi = np.mod(
            np.arctan2(y_p, x_p),
            2 * np.pi
        )

        # Convert integer environment label back to bitstring

        env_bits = format(
            e,
            f"0{len(environment)}b"
        )

        Q_S.append({
            "environment": env_bits,
            "lambda": lam,

            "x": x,
            "y": y,
            "z": z,

            "theta": theta,
            "phi": phi
        })

    return Q_S


# ============================================================
# 10. COMPLETE GQS TRAJECTORY
# ============================================================

def measure_GQS_trajectory(
    theta0,
    phi0,
    kappa,
    max_kicks,
    repetitions=10000
):
    """
    Calculate

        Q^S(1), Q^S(2), ..., Q^S(max_kicks)

    using independent experimental circuits at every time.
    """

    qubits = cirq.LineQubit.range(3)

    system = qubits[0]
    environment = qubits[1:]

    simulator = cirq.Simulator()

    Q_S_all = []

    for n in range(1, max_kicks + 1):

        print(
            f"Measuring GQS after kick {n}"
        )

        Q_n = measure_GQS_at_kick(
            simulator=simulator,
            qubits=qubits,
            system=system,
            environment=environment,
            theta0=theta0,
            phi0=phi0,
            kappa=kappa,
            nkicks=n,
            repetitions=repetitions
        )

        Q_S_all.append(Q_n)

    return Q_S_all

In [4]:
theta0 = np.pi / 2 + 0.5
phi0   = np.pi / 2

kappa = 0.5

max_kicks = 10

Q_S = measure_GQS_trajectory(
    theta0=theta0,
    phi0=phi0,
    kappa=kappa,
    max_kicks=max_kicks,
    repetitions=10000
)

Measuring GQS after kick 1
Measuring GQS after kick 2
Measuring GQS after kick 3
Measuring GQS after kick 4
Measuring GQS after kick 5
Measuring GQS after kick 6
Measuring GQS after kick 7
Measuring GQS after kick 8
Measuring GQS after kick 9
Measuring GQS after kick 10


In [5]:
Q_S

[[{'environment': '00',
   'lambda': 0.2516333333333333,
   'x': np.float64(-0.7485988791032827),
   'y': np.float64(0.6755467196819085),
   'z': np.float64(0.001577287066246047),
   'theta': np.float64(1.5692320976203902),
   'phi': np.float64(2.407445000196735)},
  {'environment': '01',
   'lambda': 0.24463333333333334,
   'x': np.float64(-0.47936248467511233),
   'y': np.float64(0.8677270824612808),
   'z': np.float64(-0.014782261286456266),
   'theta': np.float64(1.585706739461789),
   'phi': np.float64(2.0755068091052316)},
  {'environment': '10',
   'lambda': 0.25476666666666664,
   'x': np.float64(-0.46994535519125685),
   'y': np.float64(0.8696490551484766),
   'z': np.float64(0.0112540192926045),
   'theta': np.float64(1.5594119066460888),
   'phi': np.float64(2.066227667150133)},
  {'environment': '11',
   'lambda': 0.24896666666666667,
   'x': np.float64(-0.16165262735659852),
   'y': np.float64(0.9872153415900919),
   'z': np.float64(-0.002021835826930829),
   'theta': np.f